# Directed GNN + Linear Programming: Min-Cost Flow Candidate Screening

This notebook demonstrates why **edge direction can be part of the optimization semantics**, not merely a graph-format choice.

We generate directed min-cost-flow instances and compare:

1. an **undirected/symmetrized GNN baseline**, which treats incoming and outgoing connectivity as one neighborhood;
2. a **directed GNN**, which aggregates incoming and outgoing messages separately.

The full LP provides labels for arcs that carry positive optimal flow. The learned models score candidate arcs, low-scoring arcs are screened, and the reduced LP is solved again.

Evaluation focuses on:

- recall of optimal-flow arcs,
- retained-arc ratio,
- feasibility,
- objective gap,
- full vs reduced solve time.

The GNN is therefore used as a **candidate filter**, while the LP solver remains responsible for feasibility and the final solution.


In [ ]:
import random
import time
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

from scipy.optimize import linprog

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


## 1. Generate feasible directed flow networks

Each instance has a source with positive supply, a sink with matching demand, and intermediate transshipment nodes. A directed backbone guarantees at least one feasible source-to-sink path; additional random directed arcs create alternative routes.


In [ ]:
def generate_flow_instance(n_nodes=12, extra_arc_prob=0.18, seed=0):
    rng = np.random.default_rng(seed)

    source = 0
    sink = n_nodes - 1
    total_flow = float(rng.integers(8, 18))

    balance = np.zeros(n_nodes, dtype=float)
    balance[source] = total_flow
    balance[sink] = -total_flow

    arc_set = set()

    # Directed backbone: 0 -> 1 -> ... -> sink
    for u in range(n_nodes - 1):
        arc_set.add((u, u + 1))

    # Add directed shortcuts and reverse possibilities independently.
    for u in range(n_nodes):
        for v in range(n_nodes):
            if u == v:
                continue
            if rng.random() < extra_arc_prob:
                arc_set.add((u, v))

    arcs = sorted(arc_set)
    cost = []
    capacity = []

    for u, v in arcs:
        # Forward progress is usually cheaper but not always.
        progress_penalty = 0.0 if v > u else 1.5
        cost.append(float(rng.uniform(0.5, 5.0) + progress_penalty))
        capacity.append(float(rng.uniform(0.55, 1.30) * total_flow))

    # Backbone must be able to carry the required flow.
    for idx, (u, v) in enumerate(arcs):
        if v == u + 1:
            capacity[idx] = max(capacity[idx], total_flow)

    return {
        "n_nodes": n_nodes,
        "source": source,
        "sink": sink,
        "balance": balance,
        "arcs": arcs,
        "cost": np.array(cost, dtype=float),
        "capacity": np.array(capacity, dtype=float),
        "total_flow": total_flow,
    }


example = generate_flow_instance(seed=SEED)
print("nodes:", example["n_nodes"], "arcs:", len(example["arcs"]))


## 2. Solve the full min-cost-flow LP

For each directed arc \(e=(u,v)\), let \(x_e\ge 0\) be flow. We minimize

\[
\sum_e c_e x_e
\]

subject to node flow balance and arc capacities.


In [ ]:
def solve_flow(inst, allowed=None):
    m = len(inst["arcs"])

    if allowed is None:
        allowed = np.arange(m, dtype=int)
    else:
        allowed = np.array(sorted(set(map(int, allowed))), dtype=int)

    if len(allowed) == 0:
        return None

    arcs = [inst["arcs"][i] for i in allowed]
    cost = inst["cost"][allowed]
    capacity = inst["capacity"][allowed]
    n = inst["n_nodes"]

    A_eq = np.zeros((n, len(allowed)), dtype=float)

    # Convention: outgoing - incoming = balance.
    for j, (u, v) in enumerate(arcs):
        A_eq[u, j] += 1.0
        A_eq[v, j] -= 1.0

    t0 = time.perf_counter()
    res = linprog(
        cost,
        A_eq=A_eq,
        b_eq=inst["balance"],
        bounds=[(0.0, float(cap)) for cap in capacity],
        method="highs",
    )
    elapsed = time.perf_counter() - t0

    if not res.success or res.x is None:
        return None

    x = np.zeros(m, dtype=float)
    x[allowed] = res.x

    return {
        "x": x,
        "obj": float(inst["cost"] @ x),
        "time": elapsed,
    }


full_example = solve_flow(example)
print("objective:", full_example["obj"])
print("positive-flow arcs:", int(np.sum(full_example["x"] > 1e-7)))


## 3. Graph features

Node features:

- normalized supply/demand balance,
- source indicator,
- sink indicator,
- normalized in-degree,
- normalized out-degree.

Arc features:

- normalized cost,
- normalized capacity,
- normalized topological direction signal `(v-u)/(n-1)`.

The direction-aware model also receives the original directed edge index.


In [ ]:
def build_graph(inst):
    n = inst["n_nodes"]
    arcs = inst["arcs"]
    m = len(arcs)

    indeg = np.zeros(n)
    outdeg = np.zeros(n)
    for u, v in arcs:
        outdeg[u] += 1
        indeg[v] += 1

    balance = inst["balance"] / max(inst["total_flow"], 1.0)
    x = np.column_stack([
        balance,
        np.arange(n) == inst["source"],
        np.arange(n) == inst["sink"],
        indeg / max(indeg.max(), 1.0),
        outdeg / max(outdeg.max(), 1.0),
    ]).astype(np.float32)

    src = np.array([u for u, v in arcs], dtype=np.int64)
    dst = np.array([v for u, v in arcs], dtype=np.int64)

    cost = inst["cost"] / max(inst["cost"].max(), 1e-9)
    cap = inst["capacity"] / max(inst["capacity"].max(), 1e-9)
    direction = (dst - src) / max(n - 1, 1)

    edge_attr = np.column_stack([cost, cap, direction]).astype(np.float32)

    directed_edge_index = torch.tensor(
        np.vstack([src, dst]),
        dtype=torch.long,
    )

    # Symmetrized connectivity for the undirected baseline.
    undirected_edge_index = torch.tensor(
        np.hstack([
            np.vstack([src, dst]),
            np.vstack([dst, src]),
        ]),
        dtype=torch.long,
    )

    return {
        "x": torch.tensor(x),
        "directed_edge_index": directed_edge_index,
        "undirected_edge_index": undirected_edge_index,
        "arc_pairs": torch.tensor(np.vstack([src, dst]).T, dtype=torch.long),
        "edge_attr": torch.tensor(edge_attr),
    }


build_graph(example)


## 4. Message-passing layers

The directed layer computes separate incoming and outgoing aggregates. The undirected baseline uses the symmetrized graph and a single neighborhood aggregate.


In [ ]:
def mean_aggregate(messages, index, dim_size):
    out = messages.new_zeros((dim_size, messages.size(-1)))
    out.index_add_(0, index, messages)

    count = messages.new_zeros((dim_size, 1))
    count.index_add_(0, index, messages.new_ones((messages.size(0), 1)))
    return out / count.clamp_min(1.0)


class UndirectedBlock(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.msg = nn.Linear(hidden, hidden)
        self.update = nn.Sequential(
            nn.Linear(2 * hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
        )
        self.norm = nn.LayerNorm(hidden)

    def forward(self, h, edge_index):
        src, dst = edge_index
        msg = self.msg(h[src])
        agg = mean_aggregate(msg, dst, h.size(0))
        return self.norm(h + self.update(torch.cat([h, agg], dim=-1)))


class DirectedBlock(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.in_msg = nn.Linear(hidden, hidden)
        self.out_msg = nn.Linear(hidden, hidden)
        self.update = nn.Sequential(
            nn.Linear(3 * hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
        )
        self.norm = nn.LayerNorm(hidden)

    def forward(self, h, edge_index):
        src, dst = edge_index

        incoming = mean_aggregate(
            self.in_msg(h[src]),
            dst,
            h.size(0),
        )
        outgoing = mean_aggregate(
            self.out_msg(h[dst]),
            src,
            h.size(0),
        )

        return self.norm(
            h + self.update(torch.cat([h, incoming, outgoing], dim=-1))
        )


class ArcScorer(nn.Module):
    def __init__(self, directed=True, hidden=64, layers=3):
        super().__init__()
        self.directed = directed
        self.encoder = nn.Linear(5, hidden)

        block_cls = DirectedBlock if directed else UndirectedBlock
        self.blocks = nn.ModuleList([block_cls(hidden) for _ in range(layers)])

        self.head = nn.Sequential(
            nn.Linear(2 * hidden + 3, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1),
        )

    def forward(self, g):
        h = F.relu(self.encoder(g["x"]))

        if self.directed:
            edge_index = g["directed_edge_index"]
        else:
            edge_index = g["undirected_edge_index"]

        for block in self.blocks:
            h = block(h, edge_index)

        pairs = g["arc_pairs"]
        z = torch.cat([
            h[pairs[:, 0]],
            h[pairs[:, 1]],
            g["edge_attr"],
        ], dim=-1)

        return self.head(z).squeeze(-1)


def to_device(g):
    return {k: v.to(device) for k, v in g.items()}


## 5. Build the training dataset

An arc receives label 1 if the optimal full-LP solution sends positive flow through it.


In [ ]:
def build_dataset(count, start_seed):
    out = []
    seed = start_seed

    while len(out) < count:
        inst = generate_flow_instance(seed=seed)
        sol = solve_flow(inst)
        seed += 1

        if sol is None:
            continue

        g = build_graph(inst)
        y = torch.tensor((sol["x"] > 1e-7).astype(np.float32))
        out.append((inst, sol, g, y))

    return out


train_data = build_dataset(120, 1000)
test_data = build_dataset(40, 5000)

print("train:", len(train_data), "test:", len(test_data))


## 6. Train the directed and undirected models


In [ ]:
def train_model(model, epochs=70):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)

    positives = sum(float(y.sum()) for *_rest, y in train_data)
    total = sum(y.numel() for *_rest, y in train_data)
    pos_weight = torch.tensor(
        max((total - positives) / max(positives, 1.0), 1.0),
        device=device,
    )

    for epoch in range(epochs):
        order = np.random.permutation(len(train_data))
        running = 0.0

        for idx in order:
            inst, sol, g, y = train_data[idx]
            g = to_device(g)
            y = y.to(device)

            logits = model(g)
            loss = F.binary_cross_entropy_with_logits(
                logits,
                y,
                pos_weight=pos_weight,
            )

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running += float(loss)

        if (epoch + 1) % 20 == 0:
            print(
                model.__class__.__name__,
                "directed=" + str(model.directed),
                "epoch=" + str(epoch + 1),
                "loss=" + f"{running/len(train_data):.4f}",
            )

    return model


directed_model = train_model(ArcScorer(directed=True))
undirected_model = train_model(ArcScorer(directed=False))


## 7. Candidate-arc screening with adaptive fallback

If aggressive screening makes the network infeasible, retain progressively more arcs. This keeps the neural component in a **guidance** role rather than allowing it to silently break feasibility.


In [ ]:
@torch.no_grad()
def model_scores(model, g):
    model.eval()
    return torch.sigmoid(model(to_device(g))).cpu().numpy()


def cost_only_scores(inst):
    # Low cost and high capacity receive higher scores.
    s = inst["capacity"] / np.maximum(inst["cost"], 1e-9)
    return s / max(s.max(), 1e-9)


def solve_adaptive(inst, scores, initial_fraction=0.40):
    m = len(scores)

    for fraction in [initial_fraction, 0.55, 0.70, 0.85, 1.00]:
        k = max(1, int(np.ceil(fraction * m)))
        keep = np.argsort(scores)[-k:]
        sol = solve_flow(inst, keep)

        if sol is not None:
            return sol, keep, fraction

    return None, np.arange(m), 1.0


## 8. Compare optimization outcomes


In [ ]:
def evaluate(method):
    rows = []

    for inst, full, g, y in test_data:
        if method == "directed":
            scores = model_scores(directed_model, g)
        elif method == "undirected":
            scores = model_scores(undirected_model, g)
        elif method == "cost":
            scores = cost_only_scores(inst)
        else:
            raise ValueError(method)

        reduced, keep, fraction = solve_adaptive(inst, scores)

        optimal_arcs = set(np.flatnonzero(full["x"] > 1e-7).tolist())
        kept = set(map(int, keep))

        recall = len(optimal_arcs & kept) / max(len(optimal_arcs), 1)
        feasible = reduced is not None
        gap = np.nan if reduced is None else (
            100.0 * (reduced["obj"] - full["obj"]) / max(abs(full["obj"]), 1e-9)
        )

        rows.append({
            "feasible": feasible,
            "recall": recall,
            "retained": len(keep) / len(inst["arcs"]),
            "gap_pct": gap,
            "full_time": full["time"],
            "reduced_time": np.nan if reduced is None else reduced["time"],
        })

    return rows


def summarize(name, rows):
    arr = lambda k: np.array([r[k] for r in rows], dtype=float)

    return {
        "method": name,
        "feasibility": arr("feasible").mean(),
        "optimal_arc_recall": arr("recall").mean(),
        "retained_ratio": arr("retained").mean(),
        "mean_gap_pct": np.nanmean(arr("gap_pct")),
        "full_time": arr("full_time").mean(),
        "reduced_time": np.nanmean(arr("reduced_time")),
    }


summary = [
    summarize("Directed GNN", evaluate("directed")),
    summarize("Undirected/symmetrized GNN", evaluate("undirected")),
    summarize("Cost/capacity heuristic", evaluate("cost")),
]
summary


## 9. Why this is genuinely direction-sensitive

For flow optimization,

```text
u -> v
```

and

```text
v -> u
```

are different variables, with potentially different costs, capacities, and feasibility implications.

A symmetrized GNN can still be a useful baseline, but it deliberately discards a structural property of the mathematical model. The directed architecture preserves that property by maintaining separate incoming and outgoing aggregation channels.

Production extensions should include larger networks, multiple sources/sinks, time-expanded networks, distribution shift, stronger screening heuristics, and total runtime including neural inference.
